In [ ]:
# !pip install pandas numpy openpyxl json re
import pandas as pd
import numpy as np
import openpyxl as opx
# from google.colab import files
# read GoogleMap json format
import json
import re

# uploaded = files.upload()
'''已加上標籤的地點.json'''

In [2]:
def is_chinese(text):
    for char in text:
        if '\u4e00' <= char <= '\u9fff':
            return True
        return False

def round_six(value):  
    return f"{round(value, 6):.6f}"

In [46]:
# 將JSON格式轉換，建立成字典
file = '已加上標籤的地點.json'

def mk_total_km_dict(file):
    with open(file, 'r', encoding='utf-8') as file:
        json_data = json.load(file)  # 讀取為 Python 字典或列表

    total_dict ={}
    km_dict = {}
    for data in json_data['features']:
        lon = round(data['geometry']['coordinates'][0], 6)
        lat = round(data['geometry']['coordinates'][1], 6)
        loc = str(lat)+', '+str(lon)
        name = data['properties']['name']
        if is_chinese(name) == True:
            total_dict[name] = loc
        else:
            km_dict[name] = loc
    return total_dict, km_dict

total_dict = mk_total_km_dict(file)[0]
km_dict = mk_total_km_dict(file)[1]
print(total_dict)
print(km_dict)

{'玉山佛甲草': '24.116563, 121.233747', '小白頭翁': '24.175529, 121.29858', '膜蕨': '24.120051, 121.244204', '梅花草': '24.17499, 121.298372', '巒大花楸': '24.160828, 121.286657', '高山當藥': '24.120961, 121.269663', '疏花繁縷': '24.120994, 121.269813', '玉山野薔薇': '24.136433, 121.281462', '褐毛柳': '24.112333, 121.225452', '密花傅氏唐松草': '24.117941, 121.254384', '紅鞘薹': '24.121719, 121.26582', '畫眉草': '24.118583, 121.255785', '高山頭蕊蘭': '24.178358, 121.303574', '日本鹿蹄草': '24.184154, 121.341877', '三毛草': '24.164758, 121.288178', '高山青木香': '24.142084, 121.283595', '嗩吶草': '24.141901, 121.28331', '紫花地丁': '24.118358, 121.251435', '玉山捲耳': '24.164879, 121.288472', '線葉宿住薹': '24.16892, 121.292085', '玉山針藺': '24.141149, 121.272224', '高山早熟禾': '24.142664, 121.271776', '裂葉蔓黃菀': '24.114944, 121.219941', '梯牧草': '24.137271, 121.279438', '附地草': '24.119404, 121.23929', '塔塔加龍膽': '24.112025, 121.214192', '羊耳蒜': '24.120384, 121.240199', '刺花懸鉤子': '24.118536, 121.25568', '茶藨子': '24.157501, 121.285425', '高山蓼': '24.141809, 121.283198', '採土': '24.11834,

In [58]:
# 製作俗名-學名配對的list
def ver_sci_list(file_name, type_):
    with open(file_name, 'r', encoding ='utf-8') as file:
        lines = file.readlines()

    dict_list = []
    for line in lines:
        # 去除首尾空格和換行
        line = line.strip()
        # 使用正則表達式匹配 key-value
        matches = re.findall(r"'(\w+)': '([^']*)'", line)
        # 轉換為字典
        item_dict = {key: value for key, value in matches}
        dict_list.append(item_dict)

    # 顯示結果
    verName_sciName_list = []
    for item in dict_list:
        category = item.get('category')
        if category != type_:
            continue
        else:
            pass
        verName = item.get('verName')
        sciName = item.get('sciName')
        verName_sciName_list.append([verName, sciName])

    return verName_sciName_list

In [ ]:
# 使用俗名-學名配對的list，篩選出外來種點位資訊
# type_ = 8，即為挑出外來種資料
file_name = 'output_sp_result.txt'
type_ = 8
total_dict_list = []
for i in range(len(total_dict)):

    # 將字典中的座標、植物名及資料依序放入
    loc = list(total_dict.items())[i][1]
    name = list(total_dict.items())[i][0]

    ver_sci = ver_sci_list(file_name, type_)
    verName = ''
    sciName = ''
    for n in range(len(ver_sci)):
        ver = ver_sci[n][0]
        sci = ver_sci[n][1]
        if name != ver:
            continue
        else:
            verName = ver
            sciName = sci
            locality = loc
            pass
        total_dict_list.append([verName, sciName, locality])

print(total_dict_list)

[]


In [53]:
# 將包含座標、植物名及資料寫入新excel
# 由workbook創建worksheet
wb = opx.Workbook()
ws = wb.active
ws.title = '植物GoogleMap點位'
ws.cell(row=1, column=1).value = 'occurrenceID' #流水號
ws.cell(row=1, column=2).value = 'vernacularName' #俗名
ws.cell(row=1, column=3).value = 'scientificName' #學名
ws.cell(row=1, column=4).value = 'verbatimCoordinateSystem' #座標
ws.cell(row=1, column=5).value = 'eventDate' #日期

# 依字典順序讀取字典中的座標、植物名及資料
a = 2
for i in range(len(total_dict)):
    # 將字典中的座標、植物名及資料依序放入
    # verName = total_dict_list[i][0]
    # sciName = total_dict_list[i][1]
    # loc = total_dict_list[i][2]
    loc = list(total_dict.items())[i][1]
    name = list(total_dict.items())[i][0]

    ws.cell(row=a, column=2).value = verName
    ws.cell(row=a, column=3).value = sciName
    ws.cell(row=a, column=4).value = loc
    a += 1

# 儲存並下載檔案
file_name = 'GoogleMap點位SDM格式.xlsx'
wb.save(file_name)